## Regressão Logística

A [Regressão Logística](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) é um modelo de classificação supervisionada amplamente utilizado em problemas binários.

O modelo funciona a partir de uma combinação linear das variáveis de entrada, produzindo um valor que é transformado em probabilidade por meio da função logística (sigmoid), resultando em valores entre 0 e 1.

### Hiperparâmetros utilizados

- **C**: controla a regularização do modelo.
  - Valores menores (ex: 0.01) → modelo mais simples (mais regularização)
  - Valores maiores (ex: 10) → modelo mais complexo (menos regularização)
  - Os valores foram escolhidos em escala logarítmica para explorar diferentes ordens de magnitude.

- **solver**: algoritmo de otimização utilizado para encontrar os coeficientes do modelo.
  - `lbfgs`: bom desempenho geral, especialmente para datasets maiores (default)
  - `liblinear`: eficiente para problemas binários e datasets menores

### Estratégia

Foi utilizado o GridSearchCV com validação cruzada (5 folds) para identificar a melhor combinação de hiperparâmetros, utilizando a métrica ROC AUC, adequada para problemas com classes desbalanceadas.

In [1]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    import pandas as pd
    import sys
    import os

    # add raiz do projeto
    sys.path.append(os.path.abspath(".."))

    from sklearn.model_selection import GridSearchCV
    from sklearn.preprocessing import StandardScaler
    from imblearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTE
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

    from preprocessing.main_preprocessing import load_and_preprocess
    from utils.experiment_logger import save_results


save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade

🔎 Scenario: submodalidade_agrupada

🔎 Scenario: submodalidade_engineered


""


## BASELINE

In [2]:
# scenarios = [
#     "sem_submodalidade",
#     "submodalidade_agrupada",
#     "submodalidade_engineered"
# ]

# smote_options = [False, True]

# results = []

# for scenario in scenarios:
#     for use_smote in smote_options:

#         print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

#         X_train, X_test, y_train, y_test = load_and_preprocess(
#             "../predictive_models/scrdata_202505.csv",
#             scenario=scenario,
#             use_smote=use_smote
#         )

#         steps = []
#         steps.append(('scaler', StandardScaler()))
#         if use_smote:
#             steps.append(('smote', SMOTE(random_state=42)))
#         steps.append(('lr', LogisticRegression(max_iter=1000, class_weight="balanced")))

#         model = Pipeline(steps)

#         model.fit(X_train, y_train)

#         y_pred = model.predict(X_test)
#         y_proba = model.predict_proba(X_test)[:, 1]

#         results.append({
#             "model": "LogisticRegression",
#             "scenario": scenario,
#             "smote": use_smote,
#             "roc_auc": roc_auc_score(y_test, y_proba),
#             "f1": f1_score(y_test, y_pred),
#             "accuracy": accuracy_score(y_test, y_pred),
#             "n_features": X_train.shape[1],
#             "phase": "baseline",
#             "timestamp": pd.Timestamp.now()
#         })

# save_results(results, file_path="../results/experiments.csv")

# df_result = pd.DataFrame(results)

# display(df_result)


## GRIDSEARCH

In [3]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    X_train, X_test, y_train, y_test = load_and_preprocess(
        "../predictive_models/scrdata_202505.csv",
        scenario=scenario,
        use_smote=False
    )


    param_grid_lr = {
        "smote": [SMOTE(random_state=42), "passthrough"],
        "lr__C": [0.01, 0.1, 1, 10],
        "lr__solver": ["lbfgs"]
    }

    grid_lr = GridSearchCV(
        estimator=Pipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('lr', LogisticRegression(max_iter=1000,
            class_weight="balanced"))
        ]),
        param_grid=param_grid_lr,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )

    grid_lr.fit(X_train, y_train)

    print("Best parameters:", grid_lr.best_params_)
    print("Best ROC AUC (CV):", grid_lr.best_score_)


    best_lr = grid_lr.best_estimator_

    y_pred = best_lr.predict(X_test)
    y_proba = best_lr.predict_proba(X_test)[:, 1]




    cv_results = pd.DataFrame(grid_lr.cv_results_)
    cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

    for smote_val in [True, False]:
        sub_results = cv_results[cv_results['smote_used'] == smote_val]
        if not sub_results.empty:
            best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
            params = best_row['params']

            tuning_results.append({
                "model": "LogisticRegression",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "tuning_cv",
                "roc_auc": best_row['mean_test_score'],
                "f1": None,
                "accuracy": None,
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

            # Re-fit the best model for this group to get test metrics
            best_model_group = grid_lr.estimator.set_params(**params)
            best_model_group.fit(X_train, y_train)

            y_pred = best_model_group.predict(X_test)
            y_proba = best_model_group.predict_proba(X_test)[:, 1]

            tuning_results.append({
                "model": "LogisticRegression",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "test",
                "roc_auc": roc_auc_score(y_test, y_proba),
                "f1": f1_score(y_test, y_pred),
                "accuracy": accuracy_score(y_test, y_pred),
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

    save_results(tuning_results, file_path="../results/model_results.csv")

    display(pd.DataFrame(tuning_results))

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade
Best parameters: {'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': SMOTE(random_state=42)}
Best ROC AUC (CV): 0.8657405984690925


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,LogisticRegression,sem_submodalidade,True,tuning_cv,0.865741,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:09.143892
1,LogisticRegression,sem_submodalidade,True,test,0.865685,0.810887,0.791513,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:15.676780
2,LogisticRegression,sem_submodalidade,False,tuning_cv,0.865667,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:15.678452
3,LogisticRegression,sem_submodalidade,False,test,0.865526,0.810565,0.791203,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:18.923940



🔎 Scenario: submodalidade_agrupada
Best parameters: {'lr__C': 1, 'lr__solver': 'lbfgs', 'smote': 'passthrough'}
Best ROC AUC (CV): 0.9106405084790247


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,LogisticRegression,sem_submodalidade,True,tuning_cv,0.865741,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:09.143892
1,LogisticRegression,sem_submodalidade,True,test,0.865685,0.810887,0.791513,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:15.676780
2,LogisticRegression,sem_submodalidade,False,tuning_cv,0.865667,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:15.678452
3,LogisticRegression,sem_submodalidade,False,test,0.865526,0.810565,0.791203,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:18.923940
4,LogisticRegression,submodalidade_agrupada,True,tuning_cv,0.910596,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:07:20.877097
5,LogisticRegression,submodalidade_agrupada,True,test,0.911402,0.849029,0.831082,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:07:28.943946
6,LogisticRegression,submodalidade_agrupada,False,tuning_cv,0.910641,NaN,NaN,"{'lr__C': 1, 'lr__solver': 'lbfgs', 'smote': '...",2026-05-20 20:07:28.944721
7,LogisticRegression,submodalidade_agrupada,False,test,0.911407,0.847559,0.829969,"{'lr__C': 1, 'lr__solver': 'lbfgs', 'smote': '...",2026-05-20 20:07:31.724156



🔎 Scenario: submodalidade_engineered
Best parameters: {'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': SMOTE(random_state=42)}
Best ROC AUC (CV): 0.8657405984690925


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,LogisticRegression,sem_submodalidade,True,tuning_cv,0.865741,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:09.143892
1,LogisticRegression,sem_submodalidade,True,test,0.865685,0.810887,0.791513,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:15.676780
2,LogisticRegression,sem_submodalidade,False,tuning_cv,0.865667,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:15.678452
3,LogisticRegression,sem_submodalidade,False,test,0.865526,0.810565,0.791203,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:18.923940
4,LogisticRegression,submodalidade_agrupada,True,tuning_cv,0.910596,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:07:20.877097
5,LogisticRegression,submodalidade_agrupada,True,test,0.911402,0.849029,0.831082,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:07:28.943946
6,LogisticRegression,submodalidade_agrupada,False,tuning_cv,0.910641,NaN,NaN,"{'lr__C': 1, 'lr__solver': 'lbfgs', 'smote': '...",2026-05-20 20:07:28.944721
7,LogisticRegression,submodalidade_agrupada,False,test,0.911407,0.847559,0.829969,"{'lr__C': 1, 'lr__solver': 'lbfgs', 'smote': '...",2026-05-20 20:07:31.724156
8,LogisticRegression,submodalidade_engineered,True,tuning_cv,0.865741,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:08:26.887598
9,LogisticRegression,submodalidade_engineered,True,test,0.865685,0.810887,0.791513,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:08:33.406499


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,LogisticRegression,sem_submodalidade,True,tuning_cv,0.865741,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:09.143892
1,LogisticRegression,sem_submodalidade,True,test,0.865685,0.810887,0.791513,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:15.676780
2,LogisticRegression,sem_submodalidade,False,tuning_cv,0.865667,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:15.678452
3,LogisticRegression,sem_submodalidade,False,test,0.865526,0.810565,0.791203,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:06:18.923940
4,LogisticRegression,submodalidade_agrupada,True,tuning_cv,0.910596,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:07:20.877097
5,LogisticRegression,submodalidade_agrupada,True,test,0.911402,0.849029,0.831082,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:07:28.943946
6,LogisticRegression,submodalidade_agrupada,False,tuning_cv,0.910641,NaN,NaN,"{'lr__C': 1, 'lr__solver': 'lbfgs', 'smote': '...",2026-05-20 20:07:28.944721
7,LogisticRegression,submodalidade_agrupada,False,test,0.911407,0.847559,0.829969,"{'lr__C': 1, 'lr__solver': 'lbfgs', 'smote': '...",2026-05-20 20:07:31.724156
8,LogisticRegression,submodalidade_engineered,True,tuning_cv,0.865741,NaN,NaN,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:08:26.887598
9,LogisticRegression,submodalidade_engineered,True,test,0.865685,0.810887,0.791513,"{'lr__C': 10, 'lr__solver': 'lbfgs', 'smote': ...",2026-05-20 20:08:33.406499
